In [1]:
from crc import gf2mod, _crc, crc, G32, G256
import secrets

In [18]:
r4 = _crc(4, G32)
table4 = [gf2mod(1 << (i + 32 + 8), G32) for i in range(32)]

In [54]:
n1 = secrets.randbits(32)
c1 = crc(n1.to_bytes(4, "big"), G32)
i = 2
r1 = gf2mod(1 << (32 + 8 + i), G32)
c2 = crc((n1 ^ (1 << i)).to_bytes(4, "big"), G32)
print(f"c1: {c1:08x} c2: {c2:08x} c2^r1: {(c2 ^ r1):08x} c1^r1: {(c1 ^ r1):08x}")

c1: b5ea2f07 c2: f0ce0eae c2^r1: b5ea2f07 c1^r1: f0ce0eae


In [ ]:
n1 = secrets.randbits(32)
n2 = secrets.randbits(32)
c1 = crc(n1.to_bytes(4, "big"), G32)
c2 = crc(n2.to_bytes(4, "big"), G32)
c1c2 = crc((n1^n2).to_bytes(4, "big"), G32)
print(f"{c1^c2:08x} {c1c2:08x} {c1^c2^c1c2^r4:08x}")

In [ ]:
max_m1 = 0
max_m2 = 0
max_c = 0

def validate_mask(m1, m2):
    c = 0
    for i in range(256):
        if i & m1 == m2:
            if not chr(i).isalnum():
                return -1
            c += 1
    return c

for m1 in range(256):
    for m2 in range(256):
        c = validate_mask(m1, m2)
        if c > max_c:
            max_m1 = m1
            max_m2 = m2
            max_c = c

print(f"m1: {max_m1:08b} m2: {max_m2:08b} c: {max_c}")

m1: 01010001 m2: 01000001 c: 32


In [83]:
for i in range(256):
    if i & 0b1101_0001 == 0b0100_0001:
        print(chr(i), end="")

ACEGIKMOacegikmo

In [3]:
from galois import GF2
import numpy as np

In [4]:
m0 = secrets.token_bytes(64)
c0 = crc(m0, G32)
print(f"c0:{c0:08x}")

c0:3a52d89a


In [69]:
k = 32  # CRCのビット長
L = 8  # メッセージの長さ
G = G32
m1 = "A" * L
l = L.bit_length() + (-L.bit_length() % 8)
print(f"k:{k} L:{L} l:{l}")
c1 = crc(m1.encode(), G)
print(f"c1:{c1:08x}")
A_cols = []
index = []
for i in range(L * 8):
    if (1 << (i % 8)) & 0b1101_0001:
        continue
    ei = 1 << (k + l + i)
    ri = gf2mod(ei, G)
    ri_gf2 = GF2([(ri >> j) & 1 for j in range(k)])
    if np.linalg.matrix_rank(GF2(np.stack(A_cols + [ri_gf2], axis=1))) == len(A_cols) + 1:
        A_cols.append(ri_gf2)
        index.append(i)
    if len(A_cols) == k:
        break
A = GF2(np.stack(A_cols, axis=1))
print(f"rank A: {np.linalg.matrix_rank(A)}")
print(f"shape A: {A.shape}")
print(f"index: {index}")

k:32 L:8 l:8
c1:19c56e09
rank A: 32
shape A: (32, 32)
index: [1, 2, 3, 5, 9, 10, 11, 13, 17, 18, 19, 21, 25, 26, 27, 29, 33, 34, 35, 37, 41, 42, 43, 45, 49, 50, 51, 53, 57, 58, 59, 61]


In [70]:
c1_gf2 = GF2([(c1 ^ c0) >> i & 1 for i in range(k)])
ans = np.linalg.solve(A, c1_gf2)

In [71]:
m1_int = int.from_bytes(m1.encode(), "big")
for i, val in enumerate(ans):
    if val:
        m1_int ^= 1 << index[i]
m1_ans = m1_int.to_bytes(L, "big")
c1_check = crc(m1_ans, G32)
print(f"m1: {m1_ans} c1_check: {c1_check:08x} c0: {c0:08x}")

m1: b'EeCOAMIE' c1_check: 3a52d89a c0: 3a52d89a


In [50]:
f"{gf2mod(1 << (32 + 8 + 1), G32):b}"

'10100000111100101001111000001111'

In [51]:
A[:,0]

GF([1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 1, 1, 1, 1], order=2)